In [5]:
import pandas as pd

# 1. Load data starting from row index 1 to capture real headers
df = pd.read_excel('data/default of credit card clients.xls', header=1)

# 2. Rename PAY_0 to PAY_1 for chronological consistency
df.rename(columns={'PAY_0': 'PAY_1', 'default payment next month': 'default_payment_next_month'}, inplace=True)

# 3. Clean undocumented values in EDUCATION (0, 5, 6 -> 4 "Others")
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})

# 4. Clean undocumented values in MARRIAGE (0 -> 3 "Others")
df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})

# 5. Create derived analytical features for Jiwambe risk analytics
df['total_billed'] = df[['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']].sum(axis=1)
df['total_paid'] = df[['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']].sum(axis=1)
df['repayment_ratio'] = df['total_paid'] / df['total_billed'].replace(0, 1)
df['is_watchlist'] = (df['PAY_1'] > 0).astype(int)

# 6. Categorize portfolio risk buckets based on payment delays
def assign_risk(row):
    max_pay = max(row['PAY_1'], row['PAY_2'], row['PAY_3'], row['PAY_4'], row['PAY_5'], row['PAY_6'])
    if max_pay <= 0:
        return 'Low Risk (On-Time)'
    elif max_pay == 1:
        return 'Medium Risk (1 Mo Delay)'
    else:
        return 'High Risk (2+ Mo Delay)'

df['risk_category'] = df.apply(assign_risk, axis=1)

# 7. Save cleaned dataset for Power BI and BigQuery
df.to_csv('data/credit_card_clients_cleaned.csv', index=False)
print("Data cleaned successfully!")

FileNotFoundError: [Errno 2] No such file or directory: 'data/default of credit card clients.xls'

In [9]:
import pandas as pd

# 1. Load data directly from the current directory
df = pd.read_excel('default of credit card clients.xls', header=1)
df

# 2. Rename PAY_0 to PAY_1 for chronological consistency
df.rename(columns={'PAY_0': 'PAY_1', 'default payment next month': 'default_payment_next_month'}, inplace=True)

# 3. Clean undocumented values in EDUCATION (0, 5, 6 -> 4 "Others")
df['EDUCATION'] = df['EDUCATION'].replace({0: 4, 5: 4, 6: 4})

# 4. Clean undocumented values in MARRIAGE (0 -> 3 "Others")
df['MARRIAGE'] = df['MARRIAGE'].replace({0: 3})

# 5. Create derived analytical features for Jiwambe risk analytics
df['total_billed'] = df[['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']].sum(axis=1)
df['total_paid'] = df[['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']].sum(axis=1)
df['repayment_ratio'] = df['total_paid'] / df['total_billed'].replace(0, 1)
df['is_watchlist'] = (df['PAY_1'] > 0).astype(int)

# 6. Categorize portfolio risk buckets based on payment delays
def assign_risk(row):
    max_pay = max(row['PAY_1'], row['PAY_2'], row['PAY_3'], row['PAY_4'], row['PAY_5'], row['PAY_6'])
    if max_pay <= 0:
        return 'Low Risk (On-Time)'
    elif max_pay == 1:
        return 'Medium Risk (1 Mo Delay)'
    else:
        return 'High Risk (2+ Mo Delay)'

df['risk_category'] = df.apply(assign_risk, axis=1)

# 7. Save cleaned dataset as credit_card_clients_cleaned.csv
df.to_csv('credit_card_clients_cleaned.csv', index=False)
print("Data cleaned successfully! Saved credit_card_clients_cleaned.csv")

Data cleaned successfully! Saved credit_card_clients_cleaned.csv


In [11]:
import pandas as pd

# Load raw and cleaned datasets
df_raw = pd.read_excel('default of credit card clients.xls', header=1)
df_clean = pd.read_csv('credit_card_clients_cleaned.csv')

# View first 5 rows of raw data
print("--- RAW DATA HEAD ---")
print(df_raw[['ID', 'EDUCATION', 'MARRIAGE', 'PAY_0', 'default payment next month']].head())

# View first 5 rows of cleaned data
print("\n--- CLEANED DATA HEAD ---")
print(df_clean[['ID', 'EDUCATION', 'MARRIAGE', 'PAY_1', 'default_payment_next_month', 'risk_category']].head())

--- RAW DATA HEAD ---
   ID  EDUCATION  MARRIAGE  PAY_0  default payment next month
0   1          2         1      2                           1
1   2          2         2     -1                           1
2   3          2         2      0                           0
3   4          2         1      0                           0
4   5          2         1     -1                           0

--- CLEANED DATA HEAD ---
   ID  EDUCATION  MARRIAGE  PAY_1  default_payment_next_month  \
0   1          2         1      2                           1   
1   2          2         2     -1                           1   
2   3          2         2      0                           0   
3   4          2         1      0                           0   
4   5          2         1     -1                           0   

             risk_category  
0  High Risk (2+ Mo Delay)  
1  High Risk (2+ Mo Delay)  
2       Low Risk (On-Time)  
3       Low Risk (On-Time)  
4       Low Risk (On-Time)  
